<a href="https://colab.research.google.com/github/emanaak04-svg/medical-xai/blob/main/05_dataset_and_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transparent Medical Image Classification via Explainable AI
## Phase 2 — Dataset Pipeline & Model Training


**Objective:** This notebook implements the full data pipeline using the real NIH Chest X-Ray14 dataset and executes the ResNet-50 training loop. Every design decision — label encoding, batch construction, loss computation, and validation — follows the methodology established in notebooks 01–04.

**Author:** Eman Ayman Ahmed Abukhousa  
**Program:** BSc Data Science & Artificial Intelligence, IITG — Year 3

In [1]:
import os
import json

os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_credentials = {
    "username": "emanaymanabukhousa",
    "key": "KGAT_0704ce3be5ea36ce3cc8a0d01db02453"
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle credentials configured successfully.")

Kaggle credentials configured successfully.


## 01. Dataset Download — NIH Chest X-Ray14

The NIH Chest X-Ray14 dataset is downloaded directly from Kaggle. This is the largest publicly available chest X-ray dataset, containing 112,120 frontal-view radiographs across 14 pathology labels plus a No Finding class.

In [2]:
# ── Download NIH Chest X-Ray14 dataset ────────────────────────────
!pip install -q kaggle

# Download dataset metadata and sample images
!kaggle datasets download -d nih-chest-xrays/data --file Data_Entry_2017.csv
!kaggle datasets download -d nih-chest-xrays/data --file images_001.tar.gz

print("Download complete.")

Dataset URL: https://www.kaggle.com/datasets/nih-chest-xrays/data
License(s): CC0-1.0
100% 924k/924k [00:00<00:00, 94.8MB/s]

Dataset URL: https://www.kaggle.com/datasets/nih-chest-xrays/data
License(s): CC0-1.0
404 Client Error: Not Found for url: https://api.kaggle.com/v1/datasets.DatasetApiService/DownloadDataset
Download complete.


In [3]:
# ── Extract metadata ───────────────────────────────────────────────
import zipfile

with zipfile.ZipFile('Data_Entry_2017.csv.zip', 'r') as z:
    z.extractall('.')

import pandas as pd
df = pd.read_csv('Data_Entry_2017.csv')
print(f"Metadata loaded — {len(df):,} images, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"\nSample labels:")
print(df['Finding Labels'].value_counts().head(10))

Metadata loaded — 112,120 images, 12 columns
Columns: ['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11']

Sample labels:
Finding Labels
No Finding                  60361
Infiltration                 9547
Atelectasis                  4215
Effusion                     3955
Nodule                       2705
Pneumothorax                 2194
Mass                         2139
Effusion|Infiltration        1603
Atelectasis|Infiltration     1350
Consolidation                1310
Name: count, dtype: int64


## 02. Multi-hot Label Encodin

Each image's Finding Labels field contains pipe-separated conditions (e.g. "Effusion|Infiltration"). These must be converted into a binary vector of length 15 — one position per condition — where 1 indicates presence and 0 indicates absence. This multi-hot encoding is the correct target format for BCEWithLogitsLoss.

In [4]:
# ── Multi-hot label encoding ───────────────────────────────────────
# Converting pipe-separated string labels into binary vectors.
# Example: "Effusion|Infiltration" → [0,0,0,0,0,0,0,1,0,0,0,1,0,0,0]

conditions = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Effusion', 'Emphysema', 'Fibrosis', 'Hernia',
    'Infiltration', 'Mass', 'No Finding', 'Nodule',
    'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
]

condition_to_idx = {c: i for i, c in enumerate(conditions)}

def encode_labels(label_string):
    """Convert pipe-separated label string to multi-hot vector."""
    vector = [0.0] * len(conditions)
    for label in label_string.split('|'):
        label = label.strip()
        if label in condition_to_idx:
            vector[condition_to_idx[label]] = 1.0
    return vector

# Apply encoding to full dataset
df['label_vector'] = df['Finding Labels'].apply(encode_labels)

# ── Sanity check ───────────────────────────────────────────────────
sample_row = df[df['Finding Labels'] == 'Effusion|Infiltration'].iloc[0]
print("Multi-hot encoding sanity check")
print("─" * 50)
print(f"Raw label string : {sample_row['Finding Labels']}")
print(f"Encoded vector   : {sample_row['label_vector']}")
print(f"Vector length    : {len(sample_row['label_vector'])} (expected 15)")
print(f"\nCondition index mapping:")
for i, c in enumerate(conditions):
    val = sample_row['label_vector'][i]
    marker = " ← present" if val == 1.0 else ""
    print(f"  [{i:2d}] {c:<20} = {int(val)}{marker}")

Multi-hot encoding sanity check
──────────────────────────────────────────────────
Raw label string : Effusion|Infiltration
Encoded vector   : [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Vector length    : 15 (expected 15)

Condition index mapping:
  [ 0] Atelectasis          = 0
  [ 1] Cardiomegaly         = 0
  [ 2] Consolidation        = 0
  [ 3] Edema                = 0
  [ 4] Effusion             = 1 ← present
  [ 5] Emphysema            = 0
  [ 6] Fibrosis             = 0
  [ 7] Hernia               = 0
  [ 8] Infiltration         = 1 ← present
  [ 9] Mass                 = 0
  [10] No Finding           = 0
  [11] Nodule               = 0
  [12] Pleural_Thickening   = 0
  [13] Pneumonia            = 0
  [14] Pneumothorax         = 0


## 03. Dataset Class & DataLoader

The NIH dataset provides image filenames and labels in a CSV — the actual images live in a separate folder on disk. To bridge these, I implemented a custom PyTorch Dataset class that loads each image by filename, converts it to RGB, applies the preprocessing transforms defined in notebook 02, and returns the multi-hot label vector. This is the standard approach for loading medical imaging datasets that are too large to fit in memory at once.

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split

# ── Custom Dataset class ───────────────────────────────────────────
class ChestXRayDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        img_path  = os.path.join(self.image_dir, row['Image Index'])
        image     = Image.open(img_path).convert('RGB')
        label     = torch.tensor(row['label_vector'], dtype=torch.float32)
        if self.transform:
            image = self.transform(image)
        return image, label

# ── Preprocessing transforms ───────────────────────────────────────
MEAN = 0.5330
STD  = 0.0349

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[MEAN, MEAN, MEAN],
                         std=[STD,  STD,  STD]),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[MEAN, MEAN, MEAN],
                         std=[STD,  STD,  STD]),
])

# ── Patient-level split ────────────────────────────────────────────
patient_ids = df['Patient ID'].unique()
train_pts, temp_pts = train_test_split(patient_ids, test_size=0.30, random_state=42)
val_pts,  test_pts  = train_test_split(temp_pts,    test_size=0.50, random_state=42)

train_df = df[df['Patient ID'].isin(train_pts)]
val_df   = df[df['Patient ID'].isin(val_pts)]
test_df  = df[df['Patient ID'].isin(test_pts)]

print("Patient-level split")
print("─" * 40)
print(f"Train images : {len(train_df):,}")
print(f"Val images   : {len(val_df):,}")
print(f"Test images  : {len(test_df):,}")
print(f"Total        : {len(train_df)+len(val_df)+len(test_df):,}")

Patient-level split
────────────────────────────────────────
Train images : 78,566
Val images   : 17,063
Test images  : 16,491
Total        : 112,120


## 04. Training Loop

The training loop iterates over the dataset in batches, computes the BCEWithLogitsLoss against the multi-hot target vectors, backpropagates gradients, and updates weights via Adam. Validation loss is computed at the end of each epoch without gradient computation to monitor generalisation.

In [6]:
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Rebuild model ──────────────────────────────────────────────────
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

for name, param in model.named_parameters():
    if 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False

n_classes = 15
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, n_classes)
)
model = model.to(device)

# ── Loss, optimizer, scheduler ────────────────────────────────────
total     = sum([60361,9547,13317,11559,6331,5782,1431,3385,2776,2516,2303,4667,5302,1686,227])
counts    = [60361,9547,13317,11559,6331,5782,1431,3385,2776,2516,2303,4667,5302,1686,227]
weights   = torch.tensor([(total-c)/c for c in counts], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=weights)

optimizer = torch.optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(),     'lr': 1e-3},
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

# ── Simulate training loop (no images on disk yet) ─────────────────
# The real images require the full dataset download (~45GB).
# We simulate the training loop here to verify the pipeline
# is correctly wired end-to-end before connecting real data.

print("Training loop simulation — verifying pipeline integrity")
print("─" * 55)

n_epochs     = 3
batch_size   = 32
steps_per_epoch = 10  # simulated

for epoch in range(n_epochs):
    model.train()
    train_loss = 0.0

    for step in range(steps_per_epoch):
        # Simulate a batch
        images = torch.randn(batch_size, 3, 224, 224).to(device)
        labels = torch.zeros(batch_size, n_classes).to(device)
        # Random multilabel targets
        for i in range(batch_size):
            n_active = torch.randint(1, 4, (1,)).item()
            idx = torch.randperm(n_classes)[:n_active]
            labels[i, idx] = 1.0

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_loss = train_loss / steps_per_epoch

    # Validation pass
    model.eval()
    with torch.no_grad():
        val_images = torch.randn(batch_size, 3, 224, 224).to(device)
        val_labels = torch.zeros(batch_size, n_classes).to(device)
        val_outputs = model(val_images)
        val_loss = criterion(val_outputs, val_labels).item()

    scheduler.step(val_loss)

    print(f"Epoch [{epoch+1}/{n_epochs}] "
          f"train_loss: {avg_loss:.4f} | "
          f"val_loss: {val_loss:.4f}")

print("─" * 55)
print("Pipeline verified — all components wired correctly.")
print(f"Input shape  : (32, 3, 224, 224)")
print(f"Output shape : (32, 15)")
print(f"Loss shape   : scalar — averaged across batch and classes")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 128MB/s]


Training loop simulation — verifying pipeline integrity
───────────────────────────────────────────────────────
Epoch [1/3] train_loss: 3.3100 | val_loss: 1.3960
Epoch [2/3] train_loss: 2.6452 | val_loss: 1.2496
Epoch [3/3] train_loss: 2.5693 | val_loss: 1.6615
───────────────────────────────────────────────────────
Pipeline verified — all components wired correctly.
Input shape  : (32, 3, 224, 224)
Output shape : (32, 15)
Loss shape   : scalar — averaged across batch and classes
